# FingerBall v2 — Stage 1: 全フレーム検出・WiLoR推論

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage1_Drive_WiLoR_OpenVINO.ipynb)

Google Drive内の既存OpenVINO INT8モデルを使用し、動画全フレームの手検出・WiLoR生出力をCSV/NPZ・BOX動画に保存します。CPUランタイムを使用します。21点復元・関節角度・ボール追跡はStage 2以降。研究動画とモデルはGitHubへアップロードしないでください。

## 0. 環境準備

In [ ]:
%pip -q install "openvino>=2025.0,<2027" opencv-python-headless matplotlib pandas pyyaml

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## 1. 設定 — 通常はここだけ編集

In [ ]:
from pathlib import Path
import os, json, time, shutil, hashlib, sys
import numpy as np
import cv2
import openvino as ov
import matplotlib.pyplot as plt
from IPython.display import display

# Driveフォルダを名前から探す。変更後の名称に対応。
PROJECT_FOLDER_NAME = '母指回内トラッキング'

# すでにMyDriveか共有ドライブの正確な絶対パスが分かる場合のみ設定。
DRIVE_PROJECT_DIR = ''  # 例: '/content/drive/MyDrive/母指回内トラッキング'

# 動画を使う場合はパスを指定。空欄ならColabから手動アップロードを選択。
VIDEO_PATH = ''
FRAME_INDEX = 0      # 0始まり。手が大きく写っているフレームへ変更
MANUAL_BOX = None    # YOLO手検出が失敗したとき (x0,y0,x1,y1) と指定可
RIGHT_HAND = True    # 検出クラスの選択。右手ならTrue（映像の左右とは別）

# Colabローカルへコピーしたキャッシュ
CACHE_ROOT = Path('/content/fingerball_stage1_models')
RESULTS_DIR_NAME = 'FingerBall_v2_stage1_results'


## 2. Drive内のフォルダを確認する

In [ ]:
def find_project(name, supplied=''):
    if supplied:
        q=Path(supplied)
        if not q.is_dir():
            raise FileNotFoundError(f'指定したDRIVE_PROJECT_DIRが存在しません: {q}')
        return q
    # mount領域の浅い階層から探す。Drive全体に対する再帰的rglobは遅いので避ける。
    roots=[Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives'), Path('/content/drive/共有ドライブ')]
    matches=[]
    for base in roots:
        if not base.is_dir():
            continue
        candidate=base/name
        if candidate.is_dir(): matches.append(candidate)
        if base.name!='MyDrive':
            try:
                for d in base.iterdir():
                    if d.is_dir() and (d/name).is_dir():matches.append(d/name)
            except OSError:pass
    if not matches:
        raise FileNotFoundError(f'「{name}」が見つかりません。Driveのマウント先を確認しDRIVE_PROJECT_DIRに絶対パスを指定してください。')
    if len(matches)>1:
        print('同名フォルダが複数あります。先頭を採用:', matches)
    return matches[0]

project=find_project(PROJECT_FOLDER_NAME, DRIVE_PROJECT_DIR)
print('対象フォルダ:',project)
print('直下の項目:',[x.name for x in project.iterdir()])
assert (project/'thumb_pronation').is_dir(), '想定構成と異なります。フォルダを確認してください。'


## 3. モデルを特定し、Colabローカルへコピー（Driveから直接推論しない）

In [ ]:
# 実際のGoogle Driveフォルダから確認された相対パス
WILOR_REL = Path('thumb_pronation/out/bench_light/verify/int8_leftcal')
DETECTOR_REL = Path('models/_DATA/wilor/pretrained_models/detector_openvino_model_288x512')

wilor_xml = project/WILOR_REL/'wilor_core_int8_leftcal.xml'
wilor_bin = wilor_xml.with_suffix('.bin')
detector_xml = project/DETECTOR_REL/'detector.xml'
detector_bin = detector_xml.with_suffix('.bin')
detector_meta = project/DETECTOR_REL/'metadata.yaml'

required = [wilor_xml,wilor_bin,detector_xml,detector_bin,detector_meta]
for x in required: print(('OK ' if x.is_file() else 'MISSING '),x)
missing=[str(x) for x in required if not x.is_file()]
if missing:
    raise FileNotFoundError('必要ファイルがありません。パスを確認:\n'+'\n'.join(missing))

CACHE_ROOT.mkdir(exist_ok=True,parents=True)
model_dir=CACHE_ROOT/'wilor';det_dir=CACHE_ROOT/'detector'
model_dir.mkdir(exist_ok=True);det_dir.mkdir(exist_ok=True)

def copy_if_changed(src,dstdir):
    dst=dstdir/src.name
    if not dst.exists() or dst.stat().st_size!=src.stat().st_size:
        print('コピー:',src.name,round(src.stat().st_size/1e6,1),'MB')
        shutil.copy2(src,dst)
    return dst

xml_local=copy_if_changed(wilor_xml,model_dir)
_ = copy_if_changed(wilor_bin,model_dir)
det_xml_local=copy_if_changed(detector_xml,det_dir)
_ = copy_if_changed(detector_bin,det_dir)
meta_local=copy_if_changed(detector_meta,det_dir)
print('コピー完了')


## 4. OpenVINOでモデルのロード・入力出力形状の検査

In [ ]:
core=ov.Core()
print("OpenVINO:",ov.__version__,"devices:",core.available_devices)
det_model=core.read_model(str(det_xml_local));wilor_model=core.read_model(str(xml_local))
for name,model in [("Detector",det_model),("WiLoR",wilor_model)]:
    print(name)
    for i,p in enumerate(model.inputs):
        print("INPUT",i,next(iter(p.get_names()),f"input_{i}"),p.partial_shape)
    for i,p in enumerate(model.outputs):
        print("OUTPUT",i,next(iter(p.get_names()),f"output_{i}"),p.partial_shape)
DEVICE="CPU"
det_compiled=core.compile_model(det_model,DEVICE,{"PERFORMANCE_HINT":"LATENCY"})
wilor_compiled=core.compile_model(wilor_model,DEVICE,{"PERFORMANCE_HINT":"LATENCY"})
print("Compiled")

## 5. 動画を指定してプレビュー（解析は後続の全フレームセル）

In [ ]:
from google.colab import files
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError("動画が指定されませんでした")
    VIDEO_PATH=str(Path("/content")/next(iter(uploaded)))
source=Path(VIDEO_PATH)
cap=cv2.VideoCapture(str(source))
if not cap.isOpened():raise RuntimeError("動画を開けません")
total_frames=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps=float(cap.get(cv2.CAP_PROP_FPS))
ok,image=cap.read();cap.release()
if not ok:raise RuntimeError("動画が読めません")
H,W=image.shape[:2]
print("video:",source.name,"frames:",total_frames,"fps:",fps)
plt.figure(figsize=(8,4));plt.imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB));plt.axis("off");plt.show()

## 6. YOLO/OpenVINOで手の検出（検出器の推論とNMS）

In [ ]:
import yaml
meta=yaml.safe_load(meta_local.read_text(encoding='utf-8'))
print('モデルメタデータ:',meta)

# 当該引き継ぎモデルのYOLO姿勢検出器の形式に合わせた処理。
# 出力 (batch, 4+nc+nk, num_predictions) を想定。形式が異なる場合は明示的に止める。

def detector_input(image_bgr, net_h=288, net_w=512):
    h,w=image_bgr.shape[:2]
    gain=min(net_h/h,net_w/w)
    nw,nh=int(round(w*gain)),int(round(h*gain))
    left=(net_w-nw)/2;top=(net_h-nh)/2
    l=int(round(left-.1));r=int(round(left+.1));t=int(round(top-.1));btm=int(round(top+.1))
    small=cv2.resize(image_bgr,(nw,nh),interpolation=cv2.INTER_LINEAR)
    small=cv2.copyMakeBorder(small,t,btm,l,r,cv2.BORDER_CONSTANT,value=(114,114,114))
    x=np.ascontiguousarray(small[None,...,::-1].transpose(0,3,1,2),dtype=np.float32)/255.0
    return x,gain,(l,t)

def nms_xyxy(boxes,scores,iou_thr=.7):
    if len(boxes)==0:return np.array([],dtype=int)
    x1,y1,x2,y2=boxes.T
    area=np.maximum(0,x2-x1)*np.maximum(0,y2-y1)
    order=np.argsort(-scores);keep=[]
    while len(order):
        i=order[0];keep.append(int(i));rest=order[1:]
        if not len(rest):break
        iw=np.maximum(0,np.minimum(x2[i],x2[rest])-np.maximum(x1[i],x1[rest]))
        ih=np.maximum(0,np.minimum(y2[i],y2[rest])-np.maximum(y1[i],y1[rest]))
        overlap=iw*ih/(area[i]+area[rest]-iw*ih+1e-12)
        order=rest[overlap<=iou_thr]
    return np.asarray(keep,dtype=int)

shape=list(det_compiled.input(0).shape)
assert len(shape)==4 and shape[0]==1 and shape[1]==3, f'未対応の手検出器入力: {shape}'
net_h,net_w=int(shape[2]),int(shape[3])
det_x,gain,(pad_x,pad_y)=detector_input(image,net_h,net_w)
start=time.perf_counter()
det_raw=np.asarray(det_compiled([det_x])[det_compiled.output(0)])
print('推論時間:',round(time.perf_counter()-start,2),'s','出力形状:',det_raw.shape)

nc=len(meta.get('names',{0:'left',1:'right'}))
if det_raw.ndim!=3 or det_raw.shape[0]!=1:
    raise RuntimeError(f'検出器出力の解釈が必要: {det_raw.shape}')
pred=det_raw[0]
if pred.shape[0]<4+nc and pred.shape[1]>=4+nc:pred=pred.T
if pred.shape[0]<4+nc:
    raise RuntimeError(f'推定器出力がYOLO形式と一致しません: {pred.shape}')
pred=pred.T
scores_all=pred[:,4:4+nc]
conf=scores_all.max(axis=1);classes=scores_all.argmax(axis=1)
keep=conf>=.3
p,conf,classes=pred[keep],conf[keep],classes[keep]
if len(p):
    xywh=p[:,:4];boxes=np.column_stack([xywh[:,0]-xywh[:,2]/2,xywh[:,1]-xywh[:,3]/2,xywh[:,0]+xywh[:,2]/2,xywh[:,1]+xywh[:,3]/2])
    chosen=nms_xyxy(boxes+classes[:,None]*7680,conf)[:300]
    boxes,conf,classes=boxes[chosen],conf[chosen],classes[chosen]
    boxes[:,[0,2]]=(boxes[:,[0,2]]-pad_x)/gain
    boxes[:,[1,3]]=(boxes[:,[1,3]]-pad_y)/gain
    boxes[:,[0,2]]=boxes[:,[0,2]].clip(0,W)
    boxes[:,[1,3]]=boxes[:,[1,3]].clip(0,H)
else:boxes=np.zeros((0,4));conf=np.zeros(0);classes=np.zeros(0,int)
print('検出数:',len(boxes),'（右/左のラベルはメタデータ・モデル実装と照合）')
for i,(box,score,cl) in enumerate(zip(boxes,conf,classes)):
    print(i,'class=',int(cl),'score=',round(float(score),3),'box=',np.rint(box).astype(int))
vis=image.copy()
for i,(box,score,cl) in enumerate(zip(boxes,conf,classes)):
    x0,y0,x1,y1=np.rint(box).astype(int)
    cv2.rectangle(vis,(x0,y0),(x1,y1),(0,210,0),2)
    cv2.putText(vis,f'{i}: {int(cl)} {score:.2f}',(x0,max(15,y0-5)),cv2.FONT_HERSHEY_SIMPLEX,.55,(255,0,0),2)
plt.figure(figsize=(12,6));plt.imshow(cv2.cvtColor(vis,cv2.COLOR_BGR2RGB));plt.axis('off');plt.show()


## 7. 全フレーム解析：手の検出＋WiLoR生出力

STRIDE=1なら全フレーム。既存の検出器は高速投球に最適化されていないため、未検出フレームも保存します。

In [ ]:
import pandas as pd
from tqdm.auto import tqdm
STRIDE=1
CONF_THRESHOLD=0.10
output_dir=project/RESULTS_DIR_NAME/"full_video"
output_dir.mkdir(parents=True,exist_ok=True)
right_class=1 if RIGHT_HAND else 0
def make_patch(frame,box,size=256):
    h,w=frame.shape[:2]
    cx=(box[0]+box[2])/2;cy=(box[1]+box[3])/2
    side=2.5*max(box[2]-box[0],box[3]-box[1])
    if not RIGHT_HAND:
        frame=cv2.flip(frame,1);cx=w-1-cx
    M=np.array([[size/side,0,size/2-cx*size/side],[0,size/side,size/2-cy*size/side]],dtype=np.float32)
    return cv2.warpAffine(frame,M,(size,size)).astype(np.float32)
cap=cv2.VideoCapture(str(source))
rows=[];outputs=[]
for frame_idx in tqdm(range(total_frames),desc="all frames"):
    ok,frame=cap.read()
    if not ok:break
    if frame_idx%STRIDE:continue
    h,w=frame.shape[:2]
    x,gain,(px,py)=detector_input(frame,net_h,net_w)
    raw=np.asarray(det_compiled([x])[det_compiled.output(0)])
    pred=raw[0].T
    score=pred[:,4:4+nc]
    conf_all=score.max(axis=1)
    cl=score.argmax(axis=1)
    mask=(conf_all>=CONF_THRESHOLD)&(cl==right_class)
    q=pred[mask];conf=conf_all[mask]
    row={"frame":frame_idx,"time_s":frame_idx/fps if fps>0 else np.nan,
         "max_detector_conf":float(conf_all.max()),"detected":0,"wilor_ok":0}
    if len(q):
        z=q[:,:4]
        b=np.column_stack((z[:,0]-z[:,2]/2,z[:,1]-z[:,3]/2,z[:,0]+z[:,2]/2,z[:,1]+z[:,3]/2))
        k=nms_xyxy(b,conf)
        if len(k):
            idx=int(k[np.argmax(conf[k])]);box=b[idx].copy()
            box[[0,2]]=(box[[0,2]]-px)/gain
            box[[1,3]]=(box[[1,3]]-py)/gain
            box[[0,2]]=np.clip(box[[0,2]],0,w)
            box[[1,3]]=np.clip(box[[1,3]],0,h)
            if box[2]>box[0] and box[3]>box[1]:
                row.update({"detected":1,"confidence":float(conf[idx]),
                  **dict(zip(("x0","y0","x1","y1"),map(float,box)))})
                try:
                    patch=make_patch(frame,box)
                    shp=list(wilor_compiled.input(0).shape)
                    inp=patch[None] if shp[-1]==3 else patch.transpose(2,0,1)[None]
                    result=wilor_compiled([np.ascontiguousarray(inp)])
                    arr=[np.asarray(result[p]).copy() for p in wilor_compiled.outputs]
                    if all(np.all(np.isfinite(a)) for a in arr):
                        row["wilor_ok"]=1
                        outputs.append((frame_idx,arr))
                except Exception as ex:row["error"]=str(ex)
    rows.append(row)
cap.release()
df=pd.DataFrame(rows)
csv_path=output_dir/"frame_detection_results.csv"
npz_path=output_dir/"wilor_predictions.npz"
df.to_csv(csv_path,index=False)
if outputs:
    save={"frame_idx":np.array([t for t,_ in outputs])}
    for k in range(len(outputs[0][1])):
        save["output_"+str(k)]=np.concatenate([v[k] for _,v in outputs],axis=0)
    np.savez_compressed(npz_path,**save)
print("frames:",len(df),"detected:",int(df.detected.sum()),"WiLoR:",int(df.wilor_ok.sum()))

## 8. BOX重畳動画を全フレームで作成

In [ ]:
cap=cv2.VideoCapture(str(source))
out_video=output_dir/"hand_detection_full_video.mp4"
ww=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));hh=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer=cv2.VideoWriter(str(out_video),cv2.VideoWriter_fourcc(*"mp4v"),fps if fps>0 else 30,(ww,hh))
if not writer.isOpened():raise RuntimeError("動画出力不可")
records={int(r.frame):r for r in df.itertuples(index=False)}
n=0
try:
    while True:
        ok,frame=cap.read()
        if not ok:break
        r=records.get(n)
        if r is None:status,color="not analyzed",(100,100,100)
        elif not int(r.detected):status,color="no hand",(0,0,255)
        else:
            status,color="hand | WiLoR "+("OK" if int(r.wilor_ok) else "FAIL"),(0,200,0)
            box=[getattr(r,k,np.nan) for k in ("x0","y0","x1","y1")]
            if np.all(np.isfinite(box)):
                x0,y0,x1,y1=np.rint(box).astype(int)
                cv2.rectangle(frame,(x0,y0),(x1,y1),color,2)
        cv2.putText(frame,f"{n}: {status}",(10,28),cv2.FONT_HERSHEY_SIMPLEX,.65,color,2)
        writer.write(frame);n+=1
finally:
    cap.release();writer.release()
print("overlay:",out_video,"frames:",n)

## 9. 解析結果レポート

In [ ]:
from datetime import datetime
report={"source":source.name,"timestamp":datetime.now().isoformat(timespec="seconds"),
"total_frames":total_frames,"analyzed_frames":len(df),
"detected_frames":int(df.detected.sum()),"wilor_frames":int(df.wilor_ok.sum()),
"detect_rate":float(df.detected.mean()),"stride":STRIDE,"threshold":CONF_THRESHOLD,
"device":DEVICE,"files":[str(csv_path),str(npz_path) if npz_path.exists() else None,str(out_video)],
"limitations":"Approximate crop, raw WiLoR output; no 21 joints or ball analysis"}
report_path=output_dir/"stage1_full_video_report.json"
report_path.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding="utf-8")
print(json.dumps(report,ensure_ascii=False,indent=2))

## 次のStage

このStageは手のBOXとWiLoR生出力を保存します。21点・3Dメッシュ・指関節角度・ボール追跡はまだ行いません。